In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from src.data.load import load_ratings
from src.models.svd import SVDRecommender

In [2]:

ratings = load_ratings()


train, test = train_test_split(ratings, test_size=0.2, random_state=42)

print(f"Train: {train.shape}, Test: {test.shape}")

Train: (80000, 4), Test: (20000, 4)


In [3]:
import time

print("Обучение SVD...")
start_time = time.time()

svd_model = SVDRecommender(n_factors=50, n_epochs=20) 
svd_model.fit(train)

print(f"Модель обучена за {time.time() - start_time:.2f} сек.")

# Проверяем на 2000 случайных строк из теста
test_sample = test.sample(2000, random_state=42)

actuals = test_sample["rating"].tolist()
predictions = [svd_model.predict(row["user_id"], row["movie_id"]) for _, row in test_sample.iterrows()]

rmse = np.sqrt(mean_squared_error(actuals, predictions))
print(f"SVD RMSE: {rmse:.4f}")

Обучение SVD...
Модель обучена за 1.69 сек.
SVD RMSE: 0.9229


In [4]:
likes_test = test[test["rating"] >= 3.5]
user_likes = likes_test.groupby("user_id")["movie_id"].apply(set).to_dict()
user_seen = train.groupby("user_id")["movie_id"].apply(set).to_dict()
all_movies = train["movie_id"].unique().tolist()

In [5]:
import math

ndcg_weights = [1 / math.log2(i + 1) for i in range(1, 11)]

recalls = []
ndcgs = []

for user_id, true_likes in user_likes.items():
    seen = user_seen.get(user_id, set())
    candidates = [m for m in all_movies if m not in seen]
    
    recommended_movies = svd_model.recommend(user_id, top_n=10, candidate_movies=candidates)
    
    hits = len(set(recommended_movies) & true_likes)
    recall = hits / len(true_likes)
    recalls.append(recall)
    
    dcg = 0.0
    for i, movie_id in enumerate(recommended_movies):
        if movie_id in true_likes:
            dcg += ndcg_weights[i]
            
    idcg = sum(ndcg_weights[:hits])
    
    if idcg > 0:
        ndcgs.append(dcg / idcg)
    else:
        ndcgs.append(0.0)

print(f"SVD Recall@10: {np.mean(recalls):.4f}")
print(f"SVD NDCG@10:   {np.mean(ndcgs):.4f}")

SVD Recall@10: 0.0465
SVD NDCG@10:   0.2231


In [6]:
recalls_filtered = []
ndcgs_filtered = []

for user_id, true_likes in user_likes.items():
    if len(true_likes) < 5:
        continue
        
    seen = user_seen.get(user_id, set())
    candidates = [m for m in all_movies if m not in seen]
    
    recommended_movies = svd_model.recommend(user_id, top_n=10, candidate_movies=candidates)
    
    hits = len(set(recommended_movies) & true_likes)
    recalls_filtered.append(hits / len(true_likes))
    
    dcg = sum(ndcg_weights[i] for i, movie_id in enumerate(recommended_movies) if movie_id in true_likes)
    idcg = sum(ndcg_weights[:hits])
    ndcgs_filtered.append(dcg / idcg if idcg > 0 else 0.0)

print(f"SVD Recall@10 (для юзеров с 5+ лайками): {np.mean(recalls_filtered):.4f}")
print(f"SVD NDCG@10 (для юзеров с 5+ лайками):   {np.mean(ndcgs_filtered):.4f}")

SVD Recall@10 (для юзеров с 5+ лайками): 0.0557
SVD NDCG@10 (для юзеров с 5+ лайками):   0.3059


In [7]:
train_popularity = train["movie_id"].value_counts().to_dict()

recalls_hybrid = []
ndcgs_hybrid = []

for user_id, true_likes in user_likes.items():
    if len(true_likes) < 5: 
        continue
        
    seen = user_seen.get(user_id, set())
    candidates = [m for m in all_movies if m not in seen]
    
    predictions = []
    for movie_id in candidates:
        svd_score = svd_model.predict(user_id, movie_id)
        pop_score = np.log1p(train_popularity.get(movie_id, 0))
        hybrid_score = (svd_score * 0.8) + (pop_score * 0.2)
        predictions.append((movie_id, hybrid_score))
        
    predictions.sort(key=lambda x: x[1], reverse=True)
    recommended_movies = [mid for mid, score in predictions[:10]]
    
    hits = len(set(recommended_movies) & true_likes)
    recalls_hybrid.append(hits / len(true_likes))
    
    dcg = sum(ndcg_weights[i] for i, movie_id in enumerate(recommended_movies) if movie_id in true_likes)
    idcg = sum(ndcg_weights[:hits])
    ndcgs_hybrid.append(dcg / idcg if idcg > 0 else 0.0)

print(f"Hybrid Recall@10: {np.mean(recalls_hybrid):.4f}")
print(f"Hybrid NDCG@10:   {np.mean(ndcgs_hybrid):.4f}")

Hybrid Recall@10: 0.0930
Hybrid NDCG@10:   0.4222
